In [1]:
import sys
from pathlib import Path
repo_root = Path().resolve().parent  
sys.path.append(str(repo_root))

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly import express as px
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, learning_curve, LearningCurveDisplay, RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, ElasticNet,  ElasticNetCV, RidgeCV, LassoCV, SGDRegressor, LassoLarsCV, LinearRegression
import matplotlib.pyplot as plt
from src.data.Segment_Slicer import SegmentSlicer 
SegmentSlicer = SegmentSlicer()
from scipy.optimize import fsolve
from scipy.interpolate import interp1d
import pickle

# 1. Load Data

## 1.1 Load segments

In [3]:
raw_df= pd.read_parquet(repo_root / 'data' / 'processed' / 'reunion_segments_cleaned.parquet')
df= raw_df.copy()

In [4]:
print(f"📊 Dataset: {len(raw_df)} segments")
print(f"   Running: {(raw_df['activity_type']=='Run').sum()}")
print(f"   Cycling: {(raw_df['activity_type']=='Ride').sum()}")

📊 Dataset: 6341 segments
   Running: 3481
   Cycling: 2860


In [5]:
ride_df = df[df['activity_type']=='Ride'].copy()

In [6]:
sections_dict_ride = {}
for idx, row in ride_df.iterrows():
    segment_id = row['segment_id']
    # Charger altitude_profile, distance_profile, coordinates
    sections = SegmentSlicer.cut_segment(row['altitude_profile'], row['distance_profile'], row['coordinates'])
    sections_dict_ride[segment_id] = sections

## 1.3 Extract features

In [7]:
def compute_segment_time_fast(sections, segment_distance_km, lookup_dict):
    """
    Calcule le temps total RAPIDEMENT en utilisant la lookup table.
    
    Args:
        sections: Liste de dict avec 'distance' (m) et 'grade' (%)
        segment_distance_km: Distance totale du segment en km
        lookup_dict: Dictionnaire retourné par build_lookup_table_3d()
    
    Returns:
        float: Temps total en secondes
    """
    interpolator = lookup_dict['interpolator']
    
    total_time = 0.0
    
    for section in sections:
        sect_dist_km = section['distance'] / 1000  # convertir en km
        grade = section['grade']
        
        # Lookup dans la table 3D: (segment_dist, section_dist, grade) → time(s)
        time_seconds = interpolator([segment_distance_km, sect_dist_km, grade])[0]
        
        total_time += time_seconds
    
    return total_time

In [88]:
def extract_features_from_sections(sections, segment_id=None, df=None):
    """
    Extrait des features simples à partir des sections d'un segment.
    
    Input: sections (list of dict) - output de segment_slicer.cut_segment()
    Output: dict of features
    """
    
    if not sections or len(sections) == 0:
        return None
    
    # ========== Features Basiques ==========
    total_distance = sum(s['distance'] for s in sections) / 1000  # en km
    total_elevation_gain = sum(s['elevation_gain'] for s in sections)
    total_elevation_loss = sum(s['elevation_loss'] for s in sections)
    
    # Grades
    all_grades = [s['grade'] for s in sections]
    avg_grade = np.mean(all_grades)
    max_grade = max(s['max_grade'] for s in sections)
    min_grade = min(s['min_grade'] for s in sections)
    
    # ========== Features d'Ordre (Capture la Séquence) ==========
    
    # 1. Distribution des montées (early vs late)
    early_third_distance = total_distance * 1000 * 0.33
    late_third_distance = total_distance * 1000 * 0.67
    
    early_climb_gain = sum(s['elevation_gain'] for s in sections 
                           if s['start_distance'] < early_third_distance)
    late_climb_gain = sum(s['elevation_gain'] for s in sections 
                          if s['start_distance'] > late_third_distance)
    
    early_climb_ratio = early_climb_gain / (total_elevation_gain + 1e-6)
    late_climb_ratio = late_climb_gain / (total_elevation_gain + 1e-6)
    
    # 2. Grade pondéré par position (effet fatigue)
    weighted_grade = 0
    for i, s in enumerate(sections):
        position_weight = 1 + (i / len(sections)) * 0.5  # 1.0 à 1.5x
        weighted_grade += s['grade'] * position_weight * s['distance']
    weighted_grade /= (total_distance * 1000)
    
    # 3. Position de la section la plus dure
    hardest_idx = np.argmax([s['grade'] * s['distance'] for s in sections])
    hardest_section_position = hardest_idx / len(sections)  # 0 à 1
    
    # 4. Variabilité du terrain
    grade_variance = np.mean([s['grade_variance'] for s in sections])
    
    # 5. Stats par tiers du segment
    first_third = [s for s in sections if s['start_distance'] < early_third_distance]
    middle_third = [s for s in sections 
                    if early_third_distance <= s['start_distance'] <= late_third_distance]
    last_third = [s for s in sections if s['start_distance'] > late_third_distance]
    
    first_third_avg_grade = np.mean([s['grade'] for s in first_third]) if first_third else 0
    middle_third_avg_grade = np.mean([s['grade'] for s in middle_third]) if middle_third else 0
    last_third_avg_grade = np.mean([s['grade'] for s in last_third]) if last_third else 0
    
    # 6. Compter les types de sections
    n_climbs = sum(1 for s in sections if s['type'] in ['climb', 'uphill'])
    n_descents = sum(1 for s in sections if s['type'] in ['descent', 'downhill'])
    n_flats = sum(1 for s in sections if s['type'] == 'flat')

    # 7. Features lié au data leakage
    best_time = df.loc[df['segment_id'] == segment_id, 'best_time'].iloc[0]
    avg_top_10_time = df.loc[df['segment_id'] == segment_id, 'average_top_10_time'].iloc[0]
    total_effort_count = df.loc[df['segment_id'] == segment_id, 'total_effort_count'].iloc[0]
    inv_total_effort_count = 1 / (total_effort_count + 1e-6)  

    # 8. Distance par catégorie de montée
    cat_hc_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat HC') / 1000  # en km
    cat_1_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat 1') / 1000  # en km
    cat_2_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat 2') / 1000  # en km
    cat_3_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat 3') / 1000  # en km  
    cat_4_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat 4') / 1000  # en km
    
    uphill_distance = sum(s['distance'] for s in sections if s['type'] == 'uphill') / 1000  # en km
    downhill_distance = sum(s['distance'] for s in sections if s['type'] == 'downhill') / 1000  # en km
    flat_distance = sum(s['distance'] for s in sections if s['type'] == 'flat') / 1000  # en km

    # 9. Score physique
    with open('../src/models/250W_2900Ws_05_power_profile_time_lookup_table.pkl', 'rb') as f:
        lookup_dict = pickle.load(f)
    time_score = compute_segment_time_fast(sections, total_distance, lookup_dict=lookup_dict)

    
        
    
    # ========== Assembler le dictionnaire de features ==========
    features = {
        # Basiques
        'total_distance_km': total_distance,
        'total_elevation_gain': total_elevation_gain,
        'total_elevation_loss': total_elevation_loss,
        #'avg_grade': avg_grade,
        'max_grade': max_grade,
        'min_grade': min_grade,
        'grade_variance': grade_variance,
        
        # Ordre et fatigue
        'early_climb_ratio': early_climb_ratio,
        'late_climb_ratio': late_climb_ratio,
        'weighted_grade': weighted_grade,
        'hardest_section_position': hardest_section_position,
        
        # Tiers
        'first_third_avg_grade': first_third_avg_grade,
        'middle_third_avg_grade': middle_third_avg_grade,
        'last_third_avg_grade': last_third_avg_grade,
        
        # Comptages
        #'n_sections': len(sections),
        #'n_climbs': n_climbs,
        #'n_descents': n_descents,
        #'n_flats': n_flats,

        # Data leakage
        #'best_time': best_time,
        #'avg_top_10_time': avg_top_10_time,
        #'total_effort_count': total_effort_count,
        #'inv_total_effort_count': inv_total_effort_count

        # Distance par catégorie de montée
        #'cat_hc_distance_km': cat_hc_distance,
        'cat_1_distance_km': cat_1_distance,
        'cat_2_distance_km': cat_2_distance,
        'cat_3_distance_km': cat_3_distance,
        'cat_4_distance_km': cat_4_distance,
        'uphill_distance_km': uphill_distance,
        'downhill_distance_km': downhill_distance,
        'flat_distance_km': flat_distance,

        # Polynomial features 
        'flat_cat_4_interaction': flat_distance * cat_4_distance,
        'flat_cat_3_interaction': flat_distance * cat_4_distance,
        'flat_cat_2_interaction': flat_distance * cat_4_distance,
        'flat_cat_1_interaction': flat_distance * cat_4_distance,
        'flat_DH_interaction': flat_distance * downhill_distance,
        'flat_UH_interaction': flat_distance * uphill_distance,
        'UH_cat_4_interaction': uphill_distance * cat_4_distance,
        'UH_cat_3_interaction': uphill_distance * cat_3_distance,
        'UH_cat_2_interaction': uphill_distance * cat_2_distance,
        'UH_cat_1_interaction': uphill_distance * cat_1_distance,
        'DH_cat_4_interaction': downhill_distance * cat_4_distance,
        'DH_cat_3_interaction': downhill_distance * cat_3_distance,
        'DH_cat_2_interaction': downhill_distance * cat_2_distance,
        'DH_cat_1_interaction': downhill_distance * cat_1_distance,
        
        # Physics scores
        'time_score': time_score,


    }
    
    return features

In [89]:
def extract_features_for_dataframe(df, sections_dict):
    features_list = []
    valid_indices = []
    
    for idx, row in df.iterrows():
        segment_id = row['segment_id']
        
        if segment_id not in sections_dict:
            print(f"Warning: No sections found for segment {segment_id}")
            continue
        
        sections = sections_dict[segment_id]
        
        # Passer le DataFrame à la fonction
        features = extract_features_from_sections(sections, segment_id=segment_id, df=df)
        
        if features is not None:
            features_list.append(features)
            valid_indices.append(idx)
    
    features_df = pd.DataFrame(features_list, index=valid_indices)
    print(f"✓ Extracted features for {len(features_df)} / {len(df)} segments")
    
    return features_df, valid_indices

In [90]:
features_ride_df, valid_indices_ride = extract_features_for_dataframe(ride_df, sections_dict_ride)

✓ Extracted features for 2857 / 2860 segments


In [91]:
features_ride_df['segment_id'] = ride_df.loc[valid_indices_ride, 'segment_id'].values

## 1.4 Cleaning

Cleaning : Too fast top 1 rider

In [92]:
mask_too_fast_best_rider = (ride_df['best_time'] < ride_df['average_top_10_time'] / 2)
ride_df.loc[mask_too_fast_best_rider, 'best_time'] = ride_df.loc[mask_too_fast_best_rider, 'average_top_10_time'].astype(int)
print(f"🚫 Removed {(mask_too_fast_best_rider).sum()} outliers where best_time < average_top_10_time / 2")

🚫 Removed 0 outliers where best_time < average_top_10_time / 2


Cleaning : Too steep slope ( |slope|>25% )

In [93]:
mask_too_steep_slope = (features_ride_df['max_grade'] <= 30) & (features_ride_df['min_grade'] >= -30)
print(f"Removing {len(features_ride_df) - mask_too_steep_slope.sum()} outliers from training set based on grade thresholds.")
features_ride_df = features_ride_df.loc[mask_too_steep_slope]
sections_dict_ride = {k: v for k, v in sections_dict_ride.items() if k in ride_df['segment_id'].values}

Removing 451 outliers from training set based on grade thresholds.


Cleaned databases:
- sections_dict_ride
- df_ride

## 1.2 Load medium confidence T1 segments 

In [94]:
T1_MC_active_learning_ride = pd.read_csv(repo_root / 'data' / 'processed' / 'T1_active_learning_threshold_5_ride.csv')
T1_road_naming_ride = pd.read_csv(repo_root / 'data' / 'processed' / 'T1_road_naming_ride.csv')
segments_manually_labeled = pd.read_csv(repo_root / 'data' / 'processed' / 'segments_manually_labeled.csv')
T1_segments_manually_labeled_ride = segments_manually_labeled[(segments_manually_labeled['technicality'] == 1) & (segments_manually_labeled['segment_id'].isin(df[df['activity_type'] == 'Ride']['segment_id']))].copy()
not_T1_segments_manually_labeled_ride = segments_manually_labeled[(segments_manually_labeled['technicality'] != 1) & (segments_manually_labeled['segment_id'].isin(df[df['activity_type'] == 'Ride']['segment_id']))].copy()

T1_MC_ride = pd.concat([T1_MC_active_learning_ride, T1_road_naming_ride, T1_segments_manually_labeled_ride]).drop_duplicates().reset_index(drop=True)
T1_MC_ride = T1_MC_ride.drop(T1_MC_ride[T1_MC_ride['segment_id'].isin(not_T1_segments_manually_labeled_ride['segment_id'])].index).reset_index(drop=True)
print(f"Total T1 Medium Confidence Ride segments labeled: {len(T1_MC_ride)}")

Total T1 Medium Confidence Ride segments labeled: 1352


## 1.3 Spliting  

In [95]:
Train_ride, Test_ride = train_test_split(features_ride_df, test_size=0.2, random_state=42)
Train_ride, Val_ride = train_test_split(Train_ride, test_size=0.25, random_state=42)
print(f"Train: {len(Train_ride)}, Val: {len(Val_ride)}, Test: {len(Test_ride)}")

Train: 1443, Val: 481, Test: 482


In [96]:
Train_T1_MC_ride = Train_ride[Train_ride['segment_id'].isin(T1_MC_ride['segment_id'])].copy()
Val_T1_MC_ride = Val_ride[Val_ride['segment_id'].isin(T1_MC_ride['segment_id'])].copy()
Test_T1_MC_ride = Test_ride[Test_ride['segment_id'].isin(T1_MC_ride['segment_id'])].copy()
print(f"Train T1 MC: {len(Train_T1_MC_ride)}, Val T1 MC: {len(Val_T1_MC_ride)}, Test T1 MC: {len(Test_T1_MC_ride)}")

Train T1 MC: 757, Val T1 MC: 261, Test T1 MC: 274


# 2. Pipeline

## 2.1 X and Y

In [97]:
X_train_ride = Train_T1_MC_ride.drop(columns=['segment_id'])
y_train_ride = ride_df.set_index('segment_id').loc[Train_T1_MC_ride['segment_id'], 'best_time']

## 2.2 Model

In [98]:
class ModelRouter4(BaseEstimator, RegressorMixin):
    def __init__(self, pipeline_1, pipeline_2, pipeline_3, pipeline_4,
                 threshold_1=1.0, threshold_2=2.0, threshold_3=5.0, 
                 segment_length_col='segment_length',
                 drop_routing_col=True):  # ← NOUVEAU paramètre
        self.pipeline_1 = pipeline_1
        self.pipeline_2 = pipeline_2
        self.pipeline_3 = pipeline_3
        self.pipeline_4 = pipeline_4
        self.threshold_1 = threshold_1
        self.threshold_2 = threshold_2
        self.threshold_3 = threshold_3
        self.segment_length_col = segment_length_col
        self.drop_routing_col = drop_routing_col  # ← NOUVEAU

    def _prepare_features(self, X):
        """Enlève la colonne de routing des features si demandé"""
        if self.drop_routing_col and self.segment_length_col in X.columns:
            return X.drop(columns=[self.segment_length_col])
        return X

    def fit(self, X, y):
        # Calculer les masques AVANT de retirer la colonne
        self.mask_1 = X[self.segment_length_col] <= self.threshold_1
        self.mask_2 = (X[self.segment_length_col] > self.threshold_1) & \
                      (X[self.segment_length_col] <= self.threshold_2)
        self.mask_3 = (X[self.segment_length_col] > self.threshold_2) & \
                      (X[self.segment_length_col] <= self.threshold_3)
        self.mask_4 = X[self.segment_length_col] > self.threshold_3

        # Préparer les features SANS la colonne de routing
        X_features = self._prepare_features(X)

        # Entraîner chaque pipeline sur son subset
        if np.any(self.mask_1):
            self.pipeline_1.fit(X_features[self.mask_1], y[self.mask_1])
        if np.any(self.mask_2):
            self.pipeline_2.fit(X_features[self.mask_2], y[self.mask_2])
        if np.any(self.mask_3):
            self.pipeline_3.fit(X_features[self.mask_3], y[self.mask_3])
        if np.any(self.mask_4):
            self.pipeline_4.fit(X_features[self.mask_4], y[self.mask_4])
        
        return self

    def predict(self, X):
        # Calculer les masques AVANT de retirer la colonne
        mask_1 = X[self.segment_length_col] <= self.threshold_1
        mask_2 = (X[self.segment_length_col] > self.threshold_1) & \
                 (X[self.segment_length_col] <= self.threshold_2)
        mask_3 = (X[self.segment_length_col] > self.threshold_2) & \
                 (X[self.segment_length_col] <= self.threshold_3)
        mask_4 = X[self.segment_length_col] > self.threshold_3

        # Préparer les features SANS la colonne de routing
        X_features = self._prepare_features(X)

        y_pred = np.zeros(len(X))
        if np.any(mask_1):
            y_pred[mask_1] = self.pipeline_1.predict(X_features[mask_1])
        if np.any(mask_2):
            y_pred[mask_2] = self.pipeline_2.predict(X_features[mask_2])
        if np.any(mask_3):
            y_pred[mask_3] = self.pipeline_3.predict(X_features[mask_3])
        if np.any(mask_4):
            y_pred[mask_4] = self.pipeline_4.predict(X_features[mask_4])
        
        return y_pred

In [99]:
sfs = SequentialFeatureSelector(LinearRegression(), n_features_to_select='auto', direction='forward', scoring='neg_root_mean_squared_error', tol=0.1)
sfs.fit(X_train_ride.select_dtypes(include=[np.number]), y_train_ride)
selected_features_1011 = X_train_ride.select_dtypes(include=[np.number]).columns[sfs.get_support()]
if 'total_distance_km' not in selected_features_1011:
    selected_features_1011 = selected_features_1011.insert(0, 'total_distance_km')
print("Selected features:", selected_features_1011)

Selected features: Index(['total_distance_km', 'max_grade', 'grade_variance',
       'middle_third_avg_grade', 'cat_2_distance_km', 'flat_distance_km',
       'time_score'],
      dtype='object')


# 3. Deep Dive on physical scores

In [100]:
feature_physics_score = ['time_score']

In [101]:
X_physics = X_train_ride[feature_physics_score]
y_true = y_train_ride
y_pred = X_physics.values.flatten()
residuals = y_true - y_pred

# 3. Création du graphique unique
fig = go.Figure()

# Nuage de points : Réel vs Prédit
fig.add_trace(
    go.Scatter(
        x=y_true,
        y=y_pred,
        mode='markers',
        marker=dict(size=4, opacity=0.5),
        name="Physical Score",
        customdata=Train_T1_MC_ride['segment_id'],
        hovertemplate='Actual: %{x:.0f}s<br>Predicted: %{y:.0f}s<br>Segment: %{customdata}<extra></extra>'
    )
)

# Ligne de référence y=x (Prédiction parfaite)
min_val = 0
max_val = 10000

fig.add_trace(
    go.Scatter(
        x=[min_val, max_val],
        y=[min_val, max_val],
        mode='lines',
        line=dict(color='red', dash='dash', width=2),
        name="Perfect Fit",
        hoverinfo='skip'
    )
)

# Mise en page
fig.update_layout(
    height=500,
    width=700,
    title_text="Physical Score: Actual vs Predicted Time",
    xaxis_title="Actual Time (s)",
    yaxis_title="Predicted Time (s)",
    showlegend=True
)

fig.show()

## 3.1 3 scores in Lin Reg

In [102]:
X_physics = X_train_ride[feature_physics_score]
y_true = y_train_ride

# Fit un modèle simple sur chaque score individuellement
results = {}

for score_name in feature_physics_score:
    # Modèle simple: y = a * score + b
    model = LinearRegression()
    model.fit(X_physics[[score_name]], y_true)
    
    y_pred = model.predict(X_physics[[score_name]])
    
    results[score_name] = {
        'model': model,
        'predictions': y_pred,
        'residuals': y_true - y_pred,
        'mae': mean_absolute_error(y_true, y_pred),
        'r2': r2_score(y_true, y_pred),
        'coef': model.coef_[0],
        'intercept': model.intercept_
    }

# Afficher les performances
print("="*60)
print("PERFORMANCES DES SCORES PHYSIQUES")
print("="*60)
for score_name, res in results.items():
    print(f"\n{score_name}:")
    print(f"  MAE:         {res['mae']:.2f}s")
    print(f"  R²:          {res['r2']:.3f}")
    print(f"  Formule:     time = {res['coef']:.3f} × {score_name} + {res['intercept']:.1f}")

PERFORMANCES DES SCORES PHYSIQUES

time_score:
  MAE:         64.08s
  R²:          0.969
  Formule:     time = 0.999 × time_score + 27.4


## 3.4 Residuals

In [108]:
# plot residuals in function on time
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=y_true,
        y=residuals,
        mode='markers',
        marker=dict(size=4, opacity=0.5),
        name="Physical Score Residuals",
        customdata=Train_T1_MC_ride['segment_id'],
        hovertemplate='Actual: %{x:.0f}s<br>Residual: %{y:.0f}s<br>Segment: %{customdata}<extra></extra>'
    )
)
fig.add_hline(y=0, line_dash="dash", line_color="red", opacity=0.7)
fig.update_layout(
    height=500,
    width=700,
    title_text="Physical Score: Residuals vs Actual Time",
    xaxis_title="Actual Time (s)",
    yaxis_title="Residuals (s)",
    showlegend=True
)
fig.show()

## 3.5 2D-map residuals for distance and elevation

In [104]:
# Préparer les données
scatter_df = pd.DataFrame({
    'distance_km': X_train_ride['total_distance_km'].values,      # <--- Ajout de .values
    'elevation_gain': X_train_ride['total_elevation_gain'].values, # <--- Ajout de .values
    'residual': residuals.values if hasattr(residuals, 'values') else residuals, # Gestion array/series
    'abs_residual': residuals.abs().values if hasattr(residuals, 'values') else residuals.abs(),
    'actual_time': y_true.values,                                  # <--- Ajout de .values
    'predicted_time': y_pred                                       # Déjà un numpy array
})

# Plot interactif
fig = px.scatter(
    scatter_df,
    x='distance_km',
    y='elevation_gain',
    color='residual',
    size='abs_residual',
    color_continuous_scale='RdBu_r',  # Rouge = sous-estimé, Bleu = sur-estimé
    color_continuous_midpoint=0,
    hover_data={
        'distance_km': ':.1f',
        'elevation_gain': ':.0f',
        'residual': ':.0f',
        'actual_time': ':.0f',
        'predicted_time': ':.0f'
    },
    labels={
        'distance_km': 'Distance (km)',
        'elevation_gain': 'Elevation Gain (m)',
        'residual': 'Residual (s)',
        'abs_residual': 'Abs Residual'
    },
    title=f"Résidus par Distance et Elevation "
)

fig.update_layout(
    height=600,
    width=900
)

fig.show()

# Zones problématiques
print("\n" + "="*60)
print("ZONES PROBLÉMATIQUES")
print("="*60)

# Segments longs avec peu d'elevation (sous-estimés ?)
long_flat = scatter_df[(scatter_df['distance_km'] > 10) & (scatter_df['elevation_gain'] < 200)]
print(f"\nSegments longs et plats (>10km, <200m D+):")
print(f"  Count: {len(long_flat)}")
print(f"  Mean residual: {long_flat['residual'].mean():.1f}s")

# Segments courts avec beaucoup d'elevation (sur-estimés ?)
short_steep = scatter_df[(scatter_df['distance_km'] < 3) & (scatter_df['elevation_gain'] > 300)]
print(f"\nSegments courts et raides (<3km, >300m D+):")
print(f"  Count: {len(short_steep)}")
print(f"  Mean residual: {short_steep['residual'].mean():.1f}s")


ZONES PROBLÉMATIQUES

Segments longs et plats (>10km, <200m D+):
  Count: 17
  Mean residual: -14.7s

Segments courts et raides (<3km, >300m D+):
  Count: 0
  Mean residual: nans


## 3.6 By climb cat

In [105]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

# --- 1. PRÉPARATION ROBUSTE DES DONNÉES ---
# On s'assure de ne récupérer que les valeurs brutes pour éviter tout conflit d'index
# Si residuals est une Series, on prend .values, sinon on le prend tel quel (si c'est un array numpy)
res_values = residuals.values if hasattr(residuals, 'values') else residuals
pred_values = y_pred.values if hasattr(y_pred, 'values') else y_pred

# Création du DataFrame d'analyse en réinitialisant tout
analysis_df = X_train_ride.reset_index(drop=True).copy()

# Assignation directe des valeurs (ignore les index)
analysis_df['residual'] = res_values
analysis_df['predicted'] = pred_values

# Sécurité : Supprimer les lignes qui auraient pu rester vides (normalement 0 avec .values)
analysis_df = analysis_df.dropna(subset=['residual'])

# --- 2. DÉTERMINATION DES CATÉGORIES ---
def get_dominant_category(row):
    cats = {
        'Cat 1': row.get('cat_1_distance_km', 0),
        'Cat 2': row.get('cat_2_distance_km', 0),
        'Cat 3': row.get('cat_3_distance_km', 0),
        'Cat 4': row.get('cat_4_distance_km', 0),
        'Flat': row.get('flat_distance_km', 0),
        'Uphill': row.get('uphill_distance_km', 0),
        'Downhill': row.get('downhill_distance_km', 0)
    }
    if max(cats.values()) > 0:
        return max(cats, key=cats.get)
    return 'Unknown'

analysis_df['dominant_category'] = analysis_df.apply(get_dominant_category, axis=1)

# --- 3. PLOT ---
fig = go.Figure()

fig.add_trace(go.Violin(
    y=analysis_df['residual'],
    x=analysis_df['dominant_category'],
    name='Erreur du Modèle',
    box_visible=True,
    meanline_visible=True,
    points='all',
    jitter=0.05,
    pointpos=-1.8,
    line_color='black'
))

fig.update_layout(
    title="Distribution des Résidus par Catégorie de Terrain Dominante",
    yaxis_title="Résidu (Secondes)",
    xaxis_title="Type de Segment",
    height=600,
    showlegend=False
)

fig.add_hline(y=0, line_dash="dash", line_color="green", opacity=0.7)

fig.show()

## 3.7 Score correlation

In [ ]:
# Matrice de corrélation des scores
scores_df = X_train_ride[feature_physics_score].copy()
scores_df['actual_time'] = y_true

corr_matrix = scores_df.corr()

# Heatmap
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values,
    texttemplate='%{text:.3f}',
    textfont={"size": 12},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title="Corrélation entre Scores Physiques et Actual Time",
    height=500,
    width=600
)

fig.show()

print("\n" + "="*60)
print("CORRÉLATIONS AVEC ACTUAL TIME")
print("="*60)
for score in feature_physics_score:
    corr = scores_df[[score, 'actual_time']].corr().iloc[0, 1]
    print(f"{score}: {corr:.4f}")


CORRÉLATIONS AVEC ACTUAL TIME
physiological_score: nan
time_score: nan
energy_score: nan


## 3.8 Improvements

In [ ]:
print("="*60)
print("SUGGESTIONS D'AMÉLIORATION DES SCORES")
print("="*60)

# Analyser les patterns d'erreur
best_score = 'physiological_score'
residuals_analysis = residuals_df.copy()

# 1. Segments sous-estimés (actual > predicted)
underestimated = residuals_analysis[residuals_analysis[f'residual_{best_score.split("_")[0]}'] > 100]
print(f"\n1. SEGMENTS SOUS-ESTIMÉS (n={len(underestimated)}):")
print(f"   Caractéristiques moyennes:")
print(f"   - Distance: {underestimated['segment_length'].mean():.2f} km")
print(f"   - Catégorie dominante: {underestimated['dominant_category'].mode()[0] if len(underestimated) > 0 else 'N/A'}")
print(f"   → Suggestion: Augmenter le poids de ces caractéristiques dans le score")

# 2. Segments sur-estimés (actual < predicted)
overestimated = residuals_analysis[residuals_analysis[f'residual_{best_score.split("_")[0]}'] < -100]
print(f"\n2. SEGMENTS SUR-ESTIMÉS (n={len(overestimated)}):")
print(f"   Caractéristiques moyennes:")
print(f"   - Distance: {overestimated['segment_length'].mean():.2f} km")
print(f"   - Catégorie dominante: {overestimated['dominant_category'].mode()[0] if len(overestimated) > 0 else 'N/A'}")
print(f"   → Suggestion: Réduire le poids de ces caractéristiques dans le score")

# 3. Comparaison des 3 scores
print(f"\n3. COMPARAISON DES SCORES:")
for score_name in feature_physics_score:
    mae = results[score_name]['mae']
    r2 = results[score_name]['r2']
    print(f"   {score_name}: MAE={mae:.1f}s, R²={r2:.3f}")

best = min(feature_physics_score, key=lambda s: results[s]['mae'])
print(f"\n   → Meilleur score: {best}")

print("\n4. PISTES D'AMÉLIORATION:")
print("   - Ajuster les paramètres dans build_lookup_table_3d():")
print("     * max_power_time_curve() - puissance selon distance")
print("     * fatigue_factor_distance() - facteur fatigue")
print("   - Ajouter des features manquantes (descentes, virages, etc.)")
print("   - Calibrer sur segments connus (ton frère, etc.)")

SUGGESTIONS D'AMÉLIORATION DES SCORES

1. SEGMENTS SOUS-ESTIMÉS (n=109):
   Caractéristiques moyennes:
   - Distance: nan km


KeyError: 0